# Endangered Globe — Data Pipeline

Produces `animals.geojson`: one or more label Points per threatened species, with label, IUCN category, latest assessment metadata, and Wikipedia popularity. Also writes `animals-spatial.geojson`, a heavier sidecar with the dissolved source geometry for each species.

The notebook starts in **sample mode** by default: it still queries the recommended IUCN Red List API v4 flow, but only fetches 20 mammals. Switch to `RUN_MODE = "full_iucn"` only when the exact same pipeline works end-to-end on the small table.

**Steps**
1. Query latest global IUCN assessments for selected animal classes
2. Compute label points from IUCN habitat polygons or observation points
3. Wikidata SPARQL → Wikipedia article title
4. Wikimedia Pageviews API → 12-month view count
5. Assemble & export GeoJSON

In [23]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install requests pandas tqdm geopandas shapely

import os, time, json, glob, hashlib, subprocess, sys
from collections import Counter
import requests
import pandas as pd
import geopandas as gpd
from tqdm.notebook import tqdm
from shapely.geometry import mapping
from shapely.ops import unary_union

---
## 0 · Configuration

In [24]:
RUN_MODE = "sample"  # "sample" first, then "full_iucn" once the notebook is validated

IUCN_TOKEN = os.getenv("IUCN_TOKEN", "gndepHYp7B29ctoiDs8JJsq6LDDGKmBPhzVZ")  # Required for both sample and full modes
USER_AGENT = "EndangeredGlobe/1.0 (tdemareuil@gmail.com)"  # Wikimedia requires this

TARGET_CATEGORIES = ["EW", "CR", "EN", "VU", "NT", "CD"]  # CD → displayed as NT
SLEEP_WIKI = 0.15   # seconds between Wikimedia requests
SLEEP_IUCN = 0.5    # IUCN recommends >= 0.5s between calls

OUTPUT_PATH = "animals.geojson"
SPATIAL_OUTPUT_PATH = "animals-spatial.geojson"
SPATIAL_DATA_DIR = "data/shapefiles"
CLEAN_SPATIAL_TARGETS_PATH = "data/processed/iucn_target_taxa.csv"
CLEAN_SPATIAL_OUTPUT_PATH = "data/processed/iucn_spatial_clean.geojson"
RUN_SPATIAL_CLEANING = True
IUCN_RED_LIST_VERSION = "2025-2"  # Update if your spatial download uses another Red List version
IUCN_RED_LIST_VERSION_YEAR = "2025"
IUCN_DATA_LAST_UPDATED = "10 October 2025"
SPATIAL_DATA_DOWNLOAD_DATE = "14 June 2026"  # Update to the actual IUCN spatial download date
IUCN_DATASET_CITATION = f"IUCN {IUCN_RED_LIST_VERSION_YEAR}. The IUCN Red List of Threatened Species. Version {IUCN_RED_LIST_VERSION}. https://www.iucnredlist.org. Downloaded on {SPATIAL_DATA_DOWNLOAD_DATE}."
SAMPLE_LIMIT = 20
MAX_RANGE_CENTROIDS_PER_SPECIES = 10
USE_IUCN_CACHE = True
IUCN_PAGE_SIZE = 100
GLOBAL_SCOPE_CODE = 1
SAMPLE_MAX_PAGES_PER_GROUP = 2

---
## 1 · Species list

Default path: query the IUCN API for 20 mammals only, so test runs stay small and quick.

Full path: set `RUN_MODE = "full_iucn"` to use the same API flow without the sample limit.

In [ ]:
IUCN_BASE = "https://api.iucnredlist.org/api/v4"
IUCN_CACHE_DIR = "data/cache/iucn"

# Final-mode scope: these classes match the animals we can render/read well on the globe.
# Deliberately excluded by default: fish, plants, fungi, corals, molluscs, most
# other invertebrates, and insects. Fish need a separate spatial pass because
# IUCN splits them across many shapefiles and subcategories. IUCN can filter
# Insecta, but not "large insects", so enabling it would add many tiny/obscure
# species and extra API calls.
TAXONOMIC_GROUPS = [
    {"level": "class", "name": "mammalia", "taxon_class": "Mammalia"},
    {"level": "class", "name": "aves", "taxon_class": "Aves"},
    {"level": "class", "name": "reptilia", "taxon_class": "Reptilia"},
    {"level": "class", "name": "amphibia", "taxon_class": "Amphibia"},
]

SAMPLE_TAXONOMIC_GROUPS = [
    {"level": "class", "name": "mammalia", "taxon_class": "Mammalia"},
]

TAXON_GROUP_MAP = {
    "Mammalia": "Mammals",
    "Aves": "Birds",
    "Reptilia": "Other (Reptiles, Amphibians)",
    "Amphibia": "Other (Reptiles, Amphibians)",
}

CATEGORY_LABEL_TO_CODE = {
    "EXTINCT": "EX",
    "EXTINCT IN THE WILD": "EW",
    "CRITICALLY ENDANGERED": "CR",
    "ENDANGERED": "EN",
    "VULNERABLE": "VU",
    "NEAR THREATENED": "NT",
    "LEAST CONCERN": "LC",
    "DATA DEFICIENT": "DD",
    "NOT EVALUATED": "NE",
    "CONSERVATION DEPENDENT": "CD",
}
IUCN_CATEGORY_CODES = set(CATEGORY_LABEL_TO_CODE.values())


def require_iucn_token():
    """Return the configured IUCN token, or stop early with a setup error."""
    token = (IUCN_TOKEN or "").strip()
    if not token or token == "YOUR_TOKEN_HERE":
        raise RuntimeError("Set IUCN_TOKEN in the notebook or export it as an environment variable before querying IUCN.")
    return token


def iucn_cache_path(path, params):
    """Build a stable local cache filename for one IUCN request."""
    key = json.dumps({"path": path, "params": params or {}}, sort_keys=True)
    digest = hashlib.sha1(key.encode("utf-8")).hexdigest()
    return os.path.join(IUCN_CACHE_DIR, f"{digest}.json")


def iucn_get(path, params=None):
    """GET one IUCN API resource with auth, local JSON cache, and rate limiting."""
    params = {k: v for k, v in (params or {}).items() if v is not None}
    cache_path = iucn_cache_path(path, params)
    if USE_IUCN_CACHE and os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            return json.load(f)

    headers = {"Authorization": require_iucn_token(), "User-Agent": USER_AGENT}
    r = requests.get(f"{IUCN_BASE}{path}", params=params, headers=headers, timeout=45)
    if r.status_code == 401:
        raise RuntimeError("IUCN rejected the API token. Check IUCN_TOKEN.")
    r.raise_for_status()
    data = r.json()

    if USE_IUCN_CACHE:
        os.makedirs(IUCN_CACHE_DIR, exist_ok=True)
        with open(cache_path, "w", encoding="utf-8") as f:
            json.dump(data, f)
    time.sleep(SLEEP_IUCN)
    return data


def pick_path(obj, *paths):
    """Return the first non-empty nested value found in a dict-like API response."""
    for path in paths:
        cur = obj
        for key in path:
            if isinstance(cur, dict) and key in cur:
                cur = cur[key]
            else:
                cur = None
                break
        if cur not in (None, ""):
            return cur
    return None


def normalize_category(value):
    """Normalize IUCN category labels or codes to short codes like CR, EN, VU."""
    if value is None:
        return None
    text = str(value).strip().upper()
    if text in IUCN_CATEGORY_CODES:
        return text
    return CATEGORY_LABEL_TO_CODE.get(text)


def minimal_assessment_category(assessment):
    """Read any Red List category from a minimal row before fetching detail."""
    return normalize_category(
        pick_path(
            assessment,
            ("red_list_category_code",),
            ("category_code",),
            ("red_list_category", "code"),
            ("category",),
        )
    )


def bool_or_none(value):
    """Coerce API booleans represented as bools, strings, or blanks to bool/None."""
    if value in (None, ""):
        return None
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "yes", "1"}:
        return True
    if text in {"false", "no", "0"}:
        return False
    return None


def taxon_group_from_class(taxon_class):
    """Display grouping derived from the original IUCN taxonomic class."""
    return TAXON_GROUP_MAP.get(taxon_class, taxon_class or "Unknown")


def format_counter(counter):
    """Format a small Counter for readable progress summaries."""
    if not counter:
        return "none"
    return ", ".join(f"{key}: {value:,}" for key, value in counter.most_common())


def coerce_assessment_detail(data):
    """Accept small response-shape variations and return the assessment dict."""
    if isinstance(data, dict):
        return data.get("assessment") or data
    if isinstance(data, list) and data:
        return data[0]
    return {}


def extract_population_trend(detail):
    """Return the English IUCN population trend label when present."""
    return pick_path(
        detail,
        ("population_trend", "description", "en"),
        ("population_trend", "description"),
        ("population_trend",),
    )


def extract_number_of_mature_individuals(detail):
    """Return IUCN's raw Number of mature individuals value from supplementary info."""
    return pick_path(
        detail,
        ("supplementary_info", "population_size"),
        ("population_size",),
    )


def extract_estimated_area_of_occupancy(detail):
    """Return IUCN's raw Estimated Area of Occupancy value when present."""
    return pick_path(
        detail,
        ("supplementary_info", "estimated_area_of_occupancy"),
        ("estimated_area_of_occupancy",),
    )


def extract_estimated_extent_of_occurrence(detail):
    """Return IUCN's raw Estimated Extent of Occurrence value when present."""
    return pick_path(
        detail,
        ("supplementary_info", "estimated_extent_of_occurrence"),
        ("supplementary_info", "estimated_extent_of_occurence"),
        ("estimated_extent_of_occurrence",),
        ("estimated_extent_of_occurence",),
    )


def extract_common_name(taxon):
    """Extract the preferred English common name from the nested taxon object."""
    if not isinstance(taxon, dict):
        return None
    common_names = pick_path(taxon, ("common_names",), ("commonNames",), ("taxon_common_names",)) or []
    if isinstance(common_names, list):
        candidates = [item for item in common_names if isinstance(item, dict)]
        main = next((item for item in candidates if item.get("main") or item.get("primary")), None)
        main = main or (candidates[0] if candidates else None)
        if main:
            return pick_path(main, ("name",), ("common_name",), ("description", "en"))
    return pick_path(taxon, ("main_common_name",), ("common_name",))


def extract_scientific_name(taxon):
    """Extract or reconstruct the scientific name from the nested taxon object."""
    if not isinstance(taxon, dict):
        return None
    name = pick_path(taxon, ("scientific_name",), ("scientificName",), ("name",))
    if name:
        return name
    parts = [
        pick_path(taxon, ("genus_name",), ("genus",)),
        pick_path(taxon, ("species_name",), ("species",)),
        pick_path(taxon, ("infra_name",), ("subspecies",)),
    ]
    return " ".join(str(part) for part in parts if part) or None


def taxon_rank_from_taxon(taxon):
    """Return the IUCN taxon rank bucket exposed by the assessment detail."""
    if not isinstance(taxon, dict):
        return "unknown"
    if bool_or_none(taxon.get("infrarank")):
        return "infrarank"
    if bool_or_none(taxon.get("subpopulation")):
        return "subpopulation"
    if bool_or_none(taxon.get("species")):
        return "species"
    return "unknown"


def taxon_ids_from_children(children):
    """Extract integer IUCN taxon IDs from nested child taxon objects."""
    ids = []
    if not isinstance(children, list):
        return ids
    for child in children:
        if not isinstance(child, dict):
            continue
        child_id = pick_path(child, ("sis_id",), ("sis_taxon_id",), ("taxonid",), ("id",))
        try:
            ids.append(int(child_id))
        except (TypeError, ValueError):
            continue
    return sorted(set(ids))


def parent_taxonid_from_taxon(taxon):
    """Best-effort parent species ID for infrarank/subpopulation taxa."""
    if not isinstance(taxon, dict):
        return None
    for key in ["species_taxa", "parent_taxa", "parent_taxon"]:
        value = taxon.get(key)
        if isinstance(value, list) and value:
            parent_id = pick_path(value[0], ("sis_id",), ("sis_taxon_id",), ("taxonid",), ("id",))
        elif isinstance(value, dict):
            parent_id = pick_path(value, ("sis_id",), ("sis_taxon_id",), ("taxonid",), ("id",))
        else:
            parent_id = None
        try:
            return int(parent_id)
        except (TypeError, ValueError):
            continue
    return None


def replace_species_with_available_infraranks(df):
    """When child infrarank taxa are present, display them instead of parent species."""
    present_ids = set(df["taxonid"].astype(int))
    drop_parent_ids = set()
    species_with_children_missing = 0

    for row in df.itertuples(index=False):
        if getattr(row, "taxon_rank", None) != "species":
            continue
        child_ids = getattr(row, "child_infrarank_taxonids", None) or []
        if not child_ids:
            continue
        available_child_ids = set(child_ids) & present_ids
        if available_child_ids:
            drop_parent_ids.add(int(row.taxonid))
        else:
            species_with_children_missing += 1

    if drop_parent_ids:
        print(f"Taxon granularity: replacing {len(drop_parent_ids):,} parent species with fetched infrarank taxa")
        df = df[~df["taxonid"].isin(drop_parent_ids)].copy()
    if species_with_children_missing:
        print(f"Taxon granularity: {species_with_children_missing:,} species list infrarank children, but those children were not present in the fetched assessments")
    return df


def fetch_assessments_by_taxonomy(level, name, max_pages=None):
    """Fetch minimal latest global assessments for one IUCN taxonomy filter."""
    rows = []
    page = 1
    while True:
        if max_pages is not None and page > max_pages:
            break
        data = iucn_get(
            f"/taxa/{level}/{name}",
            {
                "latest": "true",
                "scope_code": GLOBAL_SCOPE_CODE,
                "page": page,
                "per_page": IUCN_PAGE_SIZE,
            },
        )
        batch = data.get("assessments", [])
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < IUCN_PAGE_SIZE:
            break
        page += 1
    return rows


def fetch_assessment_detail(assessment_id):
    """Fetch the full IUCN assessment detail for one assessment id."""
    return coerce_assessment_detail(iucn_get(f"/assessment/{assessment_id}"))


def assessment_to_species_row(assessment, taxon_class):
    """Turn one full IUCN assessment into a species row, or return a skip reason."""
    assessment_id = assessment.get("assessment_id") or assessment.get("id")
    if not assessment_id:
        return None, "missing_assessment_id"
    try:
        assessment_id_int = int(assessment_id)
    except (TypeError, ValueError):
        return None, "invalid_assessment_id"

    detail = fetch_assessment_detail(assessment_id)
    if not isinstance(detail, dict) or not detail:
        return None, "empty_or_unexpected_detail"
    taxon = detail.get("taxon") if isinstance(detail.get("taxon"), dict) else {}
    category = normalize_category(
        pick_path(
            detail,
            ("red_list_category", "code"),
            ("red_list_category", "description", "en"),
            ("red_list_category",),
            ("category",),
            ("category_code",),
        )
    )
    if category not in TARGET_CATEGORIES:
        return None, f"detail_category_out_of_scope:{category or 'unknown'}"

    taxonid = pick_path(detail, ("sis_taxon_id",), ("taxon", "sis_id"), ("taxon", "sis_taxon_id"))
    taxonid = taxonid or assessment.get("sis_taxon_id")
    if not taxonid:
        return None, "missing_taxonid"
    try:
        taxonid_int = int(taxonid)
    except (TypeError, ValueError):
        return None, "invalid_taxonid"
    scientific_name = extract_scientific_name(taxon) or pick_path(detail, ("scientific_name",)) or assessment.get("taxon_scientific_name")

    return {
        "taxonid": taxonid_int,
        "assessment_id": assessment_id_int,
        "assessment_date": pick_path(detail, ("assessment_date",)) or assessment.get("assessment_date"),
        "year_published": pick_path(detail, ("year_published",)) or assessment.get("year_published"),
        "iucn_assessment_url": pick_path(detail, ("url",)) or assessment.get("url"),
        "iucn_citation": pick_path(detail, ("citation",)) or assessment.get("citation"),
        "scientific_name": scientific_name,
        "main_common_name": extract_common_name(taxon) or extract_common_name(detail),
        "category": category,
        "population_trend": extract_population_trend(detail),
        "number_of_mature_individuals": extract_number_of_mature_individuals(detail),
        "estimated_area_of_occupancy": extract_estimated_area_of_occupancy(detail),
        "estimated_extent_of_occurrence": extract_estimated_extent_of_occurrence(detail),
        "taxon_rank": taxon_rank_from_taxon(taxon),
        "parent_taxonid": parent_taxonid_from_taxon(taxon),
        "child_infrarank_taxonids": taxon_ids_from_children(taxon.get("infrarank_taxa")),
        "taxon_class": taxon_class,
        "taxon_group": taxon_group_from_class(taxon_class),
        "iucn_has_ranges": bool_or_none(detail.get("assessment_ranges")),
        "iucn_has_points": bool_or_none(detail.get("assessment_points")),
    }, None


def fetch_iucn_species(groups, limit=None):
    """Fetch species rows for selected taxonomy groups, with optional row limit."""
    rows = []
    seen_taxa = set()
    sample_mode = limit is not None
    per_group_target = max(1, (limit + len(groups) - 1) // len(groups)) if sample_mode else None
    max_pages = SAMPLE_MAX_PAGES_PER_GROUP if sample_mode else None

    for group in groups:
        print(f"Fetching {group['taxon_class']} assessments...")
        assessments = fetch_assessments_by_taxonomy(group["level"], group["name"], max_pages=max_pages)
        if sample_mode:
            print(f"  {len(assessments):,} candidate assessments (sample pool: up to {SAMPLE_MAX_PAGES_PER_GROUP} pages × {IUCN_PAGE_SIZE} per page)")
        else:
            print(f"  {len(assessments):,} candidate assessments")

        group_rows = []
        stats = Counter()
        skipped_minimal_categories = Counter()
        skipped_after_detail_reasons = Counter()
        for assessment in tqdm(assessments, desc=f"Scan candidates: {group['taxon_class']}"):
            stats["scanned"] += 1
            minimal_category = minimal_assessment_category(assessment)
            if minimal_category is not None and minimal_category not in TARGET_CATEGORIES:
                skipped_minimal_categories[minimal_category] += 1
                continue
            stats["detail_fetches"] += 1
            row, skip_reason = assessment_to_species_row(assessment, group["taxon_class"])
            if skip_reason:
                skipped_after_detail_reasons[skip_reason] += 1
                continue
            if row["taxonid"] in seen_taxa:
                stats["duplicates"] += 1
                continue
            rows.append(row)
            group_rows.append(row)
            seen_taxa.add(row["taxonid"])
            if per_group_target and len(group_rows) >= per_group_target:
                break
        print(
            f"  kept {len(group_rows):,} {group['taxon_class']} "
            f"after scanning {stats['scanned']:,}/{len(assessments):,} candidates; "
            f"detail fetches: {stats['detail_fetches']:,}; "
            f"skipped before detail: {sum(skipped_minimal_categories.values()):,} ({format_counter(skipped_minimal_categories)}); "
            f"skipped after detail: {sum(skipped_after_detail_reasons.values()):,} ({format_counter(skipped_after_detail_reasons)}); "
            f"duplicates: {stats['duplicates']:,}"
        )

    if limit is not None:
        rows = rows[:limit]
    if not rows:
        raise RuntimeError("IUCN returned no usable taxa. Check token, filters, and API availability.")
    return pd.DataFrame(rows)


if RUN_MODE == "sample":
    df = fetch_iucn_species(groups=SAMPLE_TAXONOMIC_GROUPS, limit=SAMPLE_LIMIT)
    print(f"Sample mode: {len(df):,} mammal taxa from IUCN")
elif RUN_MODE == "full_iucn":
    df = fetch_iucn_species(groups=TAXONOMIC_GROUPS, limit=None)
    print(f"Full mode before dedup: {len(df):,} taxa from IUCN")
else:
    raise ValueError("RUN_MODE must be 'sample' or 'full_iucn'")

df.head()

In [26]:
# Quick check on available columns and category balance.
print(df.columns.tolist())
df.category.value_counts()

['taxonid', 'assessment_id', 'assessment_date', 'year_published', 'iucn_assessment_url', 'iucn_citation', 'scientific_name', 'main_common_name', 'category', 'population_trend', 'number_of_mature_individuals', 'estimated_area_of_occupancy', 'estimated_extent_of_occurrence', 'taxon_rank', 'parent_taxonid', 'child_infrarank_taxonids', 'taxon_class', 'taxon_group', 'iucn_has_ranges', 'iucn_has_points']


category
VU    8
NT    6
EN    4
CR    2
Name: count, dtype: int64

In [27]:
# Normalise: map CD → NT for display, deduplicate on taxonid, then prefer available infraranks over parent species
df["category_iucn"] = df["category"].replace("CD", "NT")
df = df.drop_duplicates(subset="taxonid").copy()
print(f"After dedup: {len(df):,}")
before_granularity = len(df)
df = replace_species_with_available_infraranks(df)
print(f"After taxon granularity rule: {len(df):,} (removed {before_granularity - len(df):,} parent species)")
df.category_iucn.value_counts()

After dedup: 20


category_iucn
VU    8
NT    6
EN    4
CR    2
Name: count, dtype: int64

### Spatial cleaning

This optional step writes the current IUCN target table, then launches `scripts/clean_spatial_data.py`. The script reads the heavy source shapefiles once and writes a smaller spatial file containing only records relevant to the current `df`.

In [28]:
target_cols = ["taxonid", "taxon_class", "category_iucn", "scientific_name"]
os.makedirs(os.path.dirname(CLEAN_SPATIAL_TARGETS_PATH), exist_ok=True)
df[target_cols].drop_duplicates().to_csv(CLEAN_SPATIAL_TARGETS_PATH, index=False)
print(f"Written target taxa: {CLEAN_SPATIAL_TARGETS_PATH}")

if RUN_SPATIAL_CLEANING:
    cmd = [
        sys.executable,
        "scripts/clean_spatial_data.py",
        "--targets", CLEAN_SPATIAL_TARGETS_PATH,
        "--input-dir", SPATIAL_DATA_DIR,
        "--output", CLEAN_SPATIAL_OUTPUT_PATH,
    ]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print(f"Skipping spatial cleaning; expecting existing file: {CLEAN_SPATIAL_OUTPUT_PATH}")

Written target taxa: data/processed/iucn_target_taxa.csv
/Users/t.boulademareuil/.venvs/experiments/bin/python scripts/clean_spatial_data.py --targets data/processed/iucn_target_taxa.csv --input-dir data/shapefiles --output data/processed/iucn_spatial_clean.geojson
Target taxa: 20
Taxon classes: Mammalia
Spatial files: 2
Loading data/shapefiles/MAMMALS/MAMMALS_PART1.shp...
  kept 23/24 target records
Loading data/shapefiles/MAMMALS/MAMMALS_PART2.shp...
  kept 18/19 target records
Written: data/processed/iucn_spatial_clean.geojson
Clean records: 41
Species with polygons: 20
Species with fallback points: 0


---
## 2 · IUCN Spatial Data — habitat polygons / observation points → centroids

Both `sample` and `full_iucn` modes use the same spatial step. The only difference is how many `taxonid`s are present in `df`. The heavy source files are cleaned in the previous cell; this section reads the smaller cleaned spatial file.

Rules:
- If a species has one contiguous range polygon, use a centroid-like point inside that shape.
- If a species has multiple disjoint range polygons, keep up to the 10 largest components and produce one point per component.
- If a species has no range polygon, fall back to the centroid of its observation points.

Download the relevant animal shapefiles from [IUCN Spatial Data Download](https://www.iucnredlist.org/resources/spatial-data-download) into `data/shapefiles/` before running the cleaning step.

Matching method:
- IUCN API rows use `taxonid` as the species/taxon identifier.
- IUCN spatial files use `id_no` for the same identifier.
- Therefore the spatial join is `spatial.id_no == df.taxonid`, not `assessment_id`.
- `MAMMALS_PART1.shp` and `MAMMALS_PART2.shp` are two chunks of the mammal polygon dataset; concatenate them, filter to target `taxonid`s, then dissolve/group by `taxonid`.
- Use `presence` as a strong priority per taxon: Extant, Probably Extant, Possibly Extant, Possibly Extinct, Presence Uncertain, Extinct.
- Ignore `origin` for centroid placement.
- Use `seasonal` as a secondary soft priority per taxon: Resident, then Breeding, then Non-breeding, then Passage, then Seasonality Uncertain.

In [ ]:
# ── Cleaned spatial input ────────────────────────────────────────────────────

if not os.path.exists(CLEAN_SPATIAL_OUTPUT_PATH):
    raise FileNotFoundError(f"Run the spatial cleaning cell first: {CLEAN_SPATIAL_OUTPUT_PATH}")

target_ids = set(df["taxonid"].astype(int))
print(f"Loading cleaned spatial data: {CLEAN_SPATIAL_OUTPUT_PATH}")
gdf_all = gpd.read_file(CLEAN_SPATIAL_OUTPUT_PATH)
if gdf_all.crs and gdf_all.crs.to_epsg() != 4326:
    gdf_all = gdf_all.to_crs(4326)

required_spatial_cols = {"taxonid", "geometry"}
missing_spatial_cols = required_spatial_cols - set(gdf_all.columns)
if missing_spatial_cols:
    raise ValueError(f"Clean spatial file is missing columns: {sorted(missing_spatial_cols)}")
for metadata_col in ["source_path", "spatial_citation", "spatial_year"]:
    if metadata_col not in gdf_all.columns:
        gdf_all[metadata_col] = None

gdf_all["taxonid"] = pd.to_numeric(gdf_all["taxonid"], errors="coerce")
gdf_all = gdf_all[gdf_all["taxonid"].notna()].copy()
gdf_all["taxonid"] = gdf_all["taxonid"].astype(int)
gdf_all = gdf_all[gdf_all["taxonid"].isin(target_ids)].copy()
print(f"Total clean spatial records matched: {len(gdf_all):,} from {gdf_all.taxonid.nunique():,} species")

In [ ]:
# Compute label points per species from polygons first, then observation points.
if gdf_all.empty:
    raise RuntimeError("No spatial records matched. Check the cleaning output and IUCN taxon IDs.")


def polygon_parts(geometry):
    """Return every polygon component inside a geometry, recursively."""
    if geometry is None or geometry.is_empty:
        return []
    if geometry.geom_type == "Polygon":
        return [geometry]
    if geometry.geom_type == "MultiPolygon":
        return [part for part in geometry.geoms if not part.is_empty]
    if geometry.geom_type == "GeometryCollection":
        parts = []
        for part in geometry.geoms:
            parts.extend(polygon_parts(part))
        return parts
    return []


def point_parts(geometry):
    """Return every observation point inside a geometry, recursively."""
    if geometry is None or geometry.is_empty:
        return []
    if geometry.geom_type == "Point":
        return [geometry]
    if geometry.geom_type == "MultiPoint":
        return [part for part in geometry.geoms if not part.is_empty]
    if geometry.geom_type == "GeometryCollection":
        parts = []
        for part in geometry.geoms:
            parts.extend(point_parts(part))
        return parts
    return []


def safe_centroid(geometry):
    """Use centroid when it falls inside the shape, otherwise a guaranteed interior point."""
    centroid = geometry.centroid
    return centroid if geometry.covers(centroid) else geometry.representative_point()


PRESENCE_PRIORITY = {1: 1, 2: 2, 3: 3, 4: 4, 6: 5, 5: 6}
PRESENCE_LABELS = {
    1: "Extant",
    2: "Probably Extant",
    3: "Possibly Extant",
    4: "Possibly Extinct",
    5: "Extinct",
    6: "Presence Uncertain",
}
SEASONAL_PRIORITY = {1: 1, 2: 2, 3: 3, 4: 4, 5: 5}
SEASONAL_LABELS = {
    1: "Resident",
    2: "Breeding",
    3: "Non-breeding",
    4: "Passage",
    5: "Seasonality Uncertain",
}


def spatial_code(value):
    """Normalize numeric IUCN spatial distribution codes."""
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def presence_priority(value):
    """Rank presence records for representative centroid placement."""
    code = spatial_code(value)
    return PRESENCE_PRIORITY.get(code, 99)


def presence_label(value):
    """Return a readable label for a selected IUCN presence code."""
    code = spatial_code(value)
    return PRESENCE_LABELS.get(code)


def seasonal_priority(value):
    """Rank seasonal records for representative centroid placement."""
    code = spatial_code(value)
    return SEASONAL_PRIORITY.get(code, 99)


def seasonal_label(value):
    """Return a readable label for a selected IUCN seasonal code."""
    code = spatial_code(value)
    return SEASONAL_LABELS.get(code)


def best_presence_records(gdf):
    """Keep records from the best available presence bucket per taxon."""
    if gdf.empty or "spatial_presence" not in gdf.columns:
        return gdf.copy()
    ranked = gdf.copy()
    ranked["_presence_priority"] = ranked["spatial_presence"].map(presence_priority)
    best = ranked.groupby("taxonid")["_presence_priority"].transform("min")
    return ranked[ranked["_presence_priority"] == best].drop(columns="_presence_priority")


def best_seasonal_records(gdf):
    """Keep records from the best available season per taxon: resident, breeding, non-breeding, passage, uncertain."""
    if gdf.empty or "spatial_seasonal" not in gdf.columns:
        return gdf.copy()
    ranked = gdf.copy()
    ranked["_seasonal_priority"] = ranked["spatial_seasonal"].map(seasonal_priority)
    best = ranked.groupby("taxonid")["_seasonal_priority"].transform("min")
    return ranked[ranked["_seasonal_priority"] == best].drop(columns="_seasonal_priority")


gdf_all["taxonid"] = gdf_all["taxonid"].astype(int)

polygon_rows = []
point_rows = []
for row in gdf_all.itertuples(index=False):
    for geom in polygon_parts(row.geometry):
        polygon_rows.append({"taxonid": row.taxonid, "geometry": geom, "source_path": row.source_path, "spatial_citation": row.spatial_citation, "spatial_year": row.spatial_year, "spatial_presence": getattr(row, "spatial_presence", None), "spatial_seasonal": getattr(row, "spatial_seasonal", None)})
    for geom in point_parts(row.geometry):
        point_rows.append({"taxonid": row.taxonid, "geometry": geom, "source_path": row.source_path, "spatial_citation": row.spatial_citation, "spatial_year": row.spatial_year, "spatial_presence": getattr(row, "spatial_presence", None), "spatial_seasonal": getattr(row, "spatial_seasonal", None)})

polygon_gdf = gpd.GeoDataFrame(polygon_rows, geometry="geometry", crs="EPSG:4326") if polygon_rows else gpd.GeoDataFrame(columns=["taxonid", "geometry", "source_path", "spatial_citation", "spatial_year", "spatial_presence", "spatial_seasonal"], geometry="geometry", crs="EPSG:4326")
point_gdf = gpd.GeoDataFrame(point_rows, geometry="geometry", crs="EPSG:4326") if point_rows else gpd.GeoDataFrame(columns=["taxonid", "geometry", "source_path", "spatial_citation", "spatial_year", "spatial_presence", "spatial_seasonal"], geometry="geometry", crs="EPSG:4326")

polygon_gdf = best_presence_records(polygon_gdf)
polygon_gdf = best_seasonal_records(polygon_gdf)
point_gdf = best_presence_records(point_gdf)
point_gdf = best_seasonal_records(point_gdf)


def first_non_empty(values):
    """Return the first non-empty value in a pandas group."""
    for value in values:
        if value is None:
            continue
        try:
            if pd.isna(value):
                continue
        except (TypeError, ValueError):
            pass
        text = str(value).strip()
        if text:
            return text
    return None


def latest_year(values):
    """Return the latest numeric year found in a pandas group, as text."""
    years = pd.to_numeric(pd.Series(values), errors="coerce").dropna()
    return str(int(years.max())) if not years.empty else None


def build_spatial_credit(citation, year):
    """Format IUCN's required spatial-data credit for one species or dataset."""
    citation = first_non_empty([citation])
    year = first_non_empty([year])
    if citation:
        prefix = citation.rstrip(". ")
        if year and year not in prefix:
            prefix = f"{prefix} {year}"
        return f"{prefix}. The IUCN Red List of Threatened Species. Version {IUCN_RED_LIST_VERSION}. https://www.iucnredlist.org. Downloaded on {SPATIAL_DATA_DOWNLOAD_DATE}."
    return IUCN_DATASET_CITATION


spatial_meta = (
    gdf_all.groupby("taxonid", as_index=False)
    .agg(
        source_paths=("source_path", lambda s: "; ".join(sorted({str(x) for x in s if pd.notna(x)}))),
        spatial_citation=("spatial_citation", first_non_empty),
        spatial_year=("spatial_year", latest_year),
    )
)
spatial_meta["iucn_dataset_citation"] = IUCN_DATASET_CITATION
spatial_meta["iucn_data_last_updated"] = IUCN_DATA_LAST_UPDATED
spatial_meta["spatial_credit"] = spatial_meta.apply(lambda row: build_spatial_credit(row.spatial_citation, row.spatial_year), axis=1)

if not polygon_gdf.empty:
    range_geometries = polygon_gdf.dissolve(by="taxonid").reset_index()[["taxonid", "geometry"]]
    range_presence = polygon_gdf.groupby("taxonid", as_index=False).agg(spatial_presence=("spatial_presence", first_non_empty))
    range_seasons = polygon_gdf.groupby("taxonid", as_index=False).agg(spatial_seasonal=("spatial_seasonal", first_non_empty))
    range_geometries = range_geometries.merge(range_presence, on="taxonid", how="left")
    range_geometries = range_geometries.merge(range_seasons, on="taxonid", how="left")
    range_geometries["spatial_presence_label"] = range_geometries["spatial_presence"].map(presence_label)
    range_geometries["spatial_seasonal_label"] = range_geometries["spatial_seasonal"].map(seasonal_label)
else:
    range_geometries = gpd.GeoDataFrame(columns=["taxonid", "geometry", "spatial_presence", "spatial_presence_label", "spatial_seasonal", "spatial_seasonal_label"], geometry="geometry", crs="EPSG:4326")

if not point_gdf.empty:
    point_geometries = point_gdf.dissolve(by="taxonid").reset_index()[["taxonid", "geometry"]]
    point_presence = point_gdf.groupby("taxonid", as_index=False).agg(spatial_presence=("spatial_presence", first_non_empty))
    point_seasons = point_gdf.groupby("taxonid", as_index=False).agg(spatial_seasonal=("spatial_seasonal", first_non_empty))
    point_geometries = point_geometries.merge(point_presence, on="taxonid", how="left")
    point_geometries = point_geometries.merge(point_seasons, on="taxonid", how="left")
    point_geometries["spatial_presence_label"] = point_geometries["spatial_presence"].map(presence_label)
    point_geometries["spatial_seasonal_label"] = point_geometries["spatial_seasonal"].map(seasonal_label)
else:
    point_geometries = gpd.GeoDataFrame(columns=["taxonid", "geometry", "spatial_presence", "spatial_presence_label", "spatial_seasonal", "spatial_seasonal_label"], geometry="geometry", crs="EPSG:4326")

centroid_rows = []
polygon_taxa = set(range_geometries["taxonid"].astype(int)) if not range_geometries.empty else set()

for row in range_geometries.itertuples(index=False):
    parts = polygon_parts(row.geometry)
    if not parts:
        continue
    part_gdf = gpd.GeoDataFrame({"geometry": parts}, geometry="geometry", crs="EPSG:4326")
    part_gdf["area_km2"] = part_gdf.to_crs(6933).area / 1e6
    total_components = len(part_gdf)
    computed_range_area_km2 = float(part_gdf["area_km2"].sum())
    top_parts = part_gdf.sort_values("area_km2", ascending=False).head(MAX_RANGE_CENTROIDS_PER_SPECIES)
    for rank, part in enumerate(top_parts.itertuples(index=False), start=1):
        point = safe_centroid(part.geometry)
        centroid_rows.append({
            "taxonid": int(row.taxonid),
            "geometry": point,
            "lon": point.x,
            "lat": point.y,
            "centroid_source": "range_polygon",
            "centroid_rank": rank,
            "centroid_count": len(top_parts),
            "range_component_count": total_components,
            "spatial_presence": row.spatial_presence,
            "spatial_presence_label": row.spatial_presence_label,
            "spatial_seasonal": row.spatial_seasonal,
            "spatial_seasonal_label": row.spatial_seasonal_label,
            "computed_range_area_km2": computed_range_area_km2,
            "computed_range_component_area_km2": float(part.area_km2),
            "range_component_area_km2": float(part.area_km2),
            "observation_point_count": None,
        })

for row in point_geometries.itertuples(index=False):
    if int(row.taxonid) in polygon_taxa:
        continue
    points = point_parts(row.geometry)
    if not points:
        continue
    point = unary_union(points).centroid
    centroid_rows.append({
        "taxonid": int(row.taxonid),
        "geometry": point,
        "lon": point.x,
        "lat": point.y,
        "centroid_source": "observation_points",
        "centroid_rank": 1,
        "centroid_count": 1,
        "range_component_count": 0,
        "spatial_presence": row.spatial_presence,
        "spatial_presence_label": row.spatial_presence_label,
        "spatial_seasonal": row.spatial_seasonal,
        "spatial_seasonal_label": row.spatial_seasonal_label,
        "computed_range_area_km2": None,
        "computed_range_component_area_km2": None,
        "range_component_area_km2": None,
        "observation_point_count": len(points),
    })

centroids = gpd.GeoDataFrame(centroid_rows, geometry="geometry", crs="EPSG:4326")
if centroids.empty:
    raise RuntimeError("No range polygons or observation points matched the selected IUCN taxon IDs.")

spatial_geometries = pd.concat([
    range_geometries.assign(spatial_source="range_polygons"),
    point_geometries.assign(spatial_source="observation_points"),
], ignore_index=True)
spatial_geometries = gpd.GeoDataFrame(spatial_geometries, geometry="geometry", crs="EPSG:4326")

print(f"Range polygon species matched: {len(range_geometries):,}")
print(f"Observation point species matched: {len(point_geometries):,}")
print(f"Label points computed: {len(centroids):,}")

# Merge back into main df. This intentionally duplicates species with several
# disjoint range components, giving one output point per selected component.
df["taxonid"] = df["taxonid"].astype(int)
centroid_cols = [
    "taxonid", "lon", "lat", "centroid_source", "centroid_rank", "centroid_count",
    "range_component_count", "spatial_presence", "spatial_presence_label",
    "spatial_seasonal", "spatial_seasonal_label",
    "computed_range_area_km2", "computed_range_component_area_km2",
    "range_component_area_km2", "observation_point_count",
]
df = df.merge(centroids[centroid_cols], on="taxonid", how="inner")
df = df.merge(spatial_meta, on="taxonid", how="left")
print(f"Output label points with spatial data: {len(df):,} from {df.taxonid.nunique():,} species")

---
## 3 · Wikidata SPARQL — IUCN ID → Wikipedia article title

Both modes query Wikidata from IUCN taxon IDs. In sample mode, the batch is simply much smaller.

In [ ]:
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
import urllib.parse
WIKIPEDIA_LANGUAGE_PRIORITY = ["en", "de", "fr", "ja", "ru", "es", "it", "zh", "pl", "pt"]
WIKIPEDIA_LANGUAGE_RANK = {lang: rank for rank, lang in enumerate(WIKIPEDIA_LANGUAGE_PRIORITY)}


def build_sparql_query(iucn_ids):
    """Batch SPARQL: resolve IUCN taxon IDs to Wikipedia sitelinks and Wikidata images."""
    values = " ".join(f'"{i}"' for i in iucn_ids)
    return f"""
SELECT ?iucn_id ?article ?article_lang ?wiki_project ?article_title ?wikidata_image_url WHERE {{
  VALUES ?iucn_id {{ {values} }}
  ?taxon wdt:P627 ?iucn_id .          # P627 = IUCN taxon ID
  OPTIONAL {{ ?taxon wdt:P18 ?wikidata_image_url . }} # P18 = image
  ?article schema:about ?taxon ;
            schema:inLanguage ?article_lang ;
            schema:isPartOf ?wiki_site .
  FILTER(CONTAINS(STR(?wiki_site), ".wikipedia.org/"))
  BIND(REPLACE(STR(?wiki_site), "^https?://", "") AS ?wiki_project_slash)
  BIND(REPLACE(?wiki_project_slash, "/$", "") AS ?wiki_project)
  BIND(REPLACE(STR(?article), CONCAT("https://", ?wiki_project, "/wiki/"), "") AS ?article_title)
}}
"""


def article_rank(article_lang):
    """Rank Wikipedia languages; any unlisted language remains usable after the preferred list."""
    return WIKIPEDIA_LANGUAGE_RANK.get(str(article_lang), len(WIKIPEDIA_LANGUAGE_PRIORITY))

def query_wikidata_batch(iucn_ids, batch_size=500):
    """Run SPARQL in batches to avoid query size limits."""
    mapping = {}
    ids = list(map(str, iucn_ids))
    for i in tqdm(range(0, len(ids), batch_size), desc="Wikidata batches"):
        batch = ids[i:i+batch_size]
        sparql = build_sparql_query(batch)
        r = requests.get(
            WIKIDATA_ENDPOINT,
            params={"query": sparql, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=60
        )
        r.raise_for_status()
        for row in r.json()["results"]["bindings"]:
            iid = row["iucn_id"]["value"]
            title = urllib.parse.unquote(row["article_title"]["value"])
            lang = row["article_lang"]["value"]
            project = row["wiki_project"]["value"]
            article_url = row["article"]["value"]
            rank = article_rank(lang)
            entry = mapping.setdefault(iid, {"wiki_title": title, "wiki_language": lang, "wiki_project": project, "wiki_url": article_url, "wiki_rank": rank, "wikidata_image_url": None})
            if rank < entry.get("wiki_rank", 999):
                entry.update({"wiki_title": title, "wiki_language": lang, "wiki_project": project, "wiki_url": article_url, "wiki_rank": rank})
            image = row.get("wikidata_image_url", {}).get("value")
            if image and not entry["wikidata_image_url"]:
                entry["wikidata_image_url"] = image.replace("http://", "https://", 1)
        time.sleep(1.0)  # Wikidata rate limit: be gentle
    return mapping

wikidata_map = query_wikidata_batch(df["taxonid"].drop_duplicates().tolist())
print(f"Wikipedia articles found: {len(wikidata_map):,} / {len(df):,}")
print(f"Wikidata images found: {sum(1 for item in wikidata_map.values() if item.get('wikidata_image_url')):,}")
pd.Series([item.get("wiki_language") for item in wikidata_map.values()]).value_counts().head(10)

In [ ]:
# Attach wiki title and Wikidata P18 image; drop taxa with no Wikipedia article (no popularity signal)
df["wiki_title"] = df["taxonid"].astype(str).map(lambda taxonid: (wikidata_map.get(taxonid) or {}).get("wiki_title"))
df["wiki_language"] = df["taxonid"].astype(str).map(lambda taxonid: (wikidata_map.get(taxonid) or {}).get("wiki_language"))
df["wiki_project"] = df["taxonid"].astype(str).map(lambda taxonid: (wikidata_map.get(taxonid) or {}).get("wiki_project"))
df["wiki_url"] = df["taxonid"].astype(str).map(lambda taxonid: (wikidata_map.get(taxonid) or {}).get("wiki_url"))
df["wikidata_image_url"] = df["taxonid"].astype(str).map(lambda taxonid: (wikidata_map.get(taxonid) or {}).get("wikidata_image_url"))
df_wiki = df.dropna(subset=["wiki_title"]).copy()
print(f"Remaining after wiki filter: {len(df_wiki):,}")

---
## 4 · Wikimedia Pageviews API — 12-month view count

Endpoint: `https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{project}/{access}/{agent}/{article}/monthly/{start}/{end}`

We use the last 12 completed months and sum the monthly totals.

In [ ]:
from datetime import date
import urllib.parse

# Last 12 completed months
today = date.today()
end_month   = date(today.year, today.month, 1)  # first of current month
start_month = date(end_month.year - 1, end_month.month, 1)
START = start_month.strftime("%Y%m01")
END   = (date(end_month.year, end_month.month, 1) - pd.DateOffset(months=1)).strftime("%Y%m01")
# Use simpler string math to avoid pandas offset in pure Python
end_month_prev = end_month.replace(day=1)
if end_month_prev.month == 1:
    end_month_prev = end_month_prev.replace(year=end_month_prev.year-1, month=12)
else:
    end_month_prev = end_month_prev.replace(month=end_month_prev.month-1)
END = end_month_prev.strftime("%Y%m01")

print(f"Pageview window: {START} → {END}")

PAGEVIEWS_BASE = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"
WIKIPEDIA_SUMMARY_URL = "https://{project}/api/rest_v1/page/summary/{title}"

def get_pageviews(project, title):
    """Return total Wikipedia views over the last 12 months for one project/title, or 0 on error."""
    encoded = urllib.parse.quote(title, safe="")
    project = project or "en.wikipedia.org"
    url = f"{PAGEVIEWS_BASE}/{project}/all-access/all-agents/{encoded}/monthly/{START}/{END}"
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=15)
        if r.status_code == 404:
            return 0
        r.raise_for_status()
        items = r.json().get("items", [])
        return sum(item["views"] for item in items)
    except Exception:
        return 0


def get_wikipedia_thumbnail(project, title):
    """Return a Wikipedia thumbnail/original image URL for a page title, or None."""
    encoded = urllib.parse.quote(title, safe="")
    project = project or "en.wikipedia.org"
    url = WIKIPEDIA_SUMMARY_URL.format(project=project, title=encoded)
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=15)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        data = r.json()
        image = pick_path(data, ("originalimage", "source"), ("thumbnail", "source"))
        return image.replace("http://", "https://", 1) if image else None
    except Exception:
        return None

# Run once per unique article, then map back to each output label point.
article_keys = df_wiki[["wiki_project", "wiki_title"]].drop_duplicates()
pageview_map = {}
for row in tqdm(article_keys.itertuples(index=False), total=len(article_keys), desc="Pageviews"):
    pageview_map[(row.wiki_project, row.wiki_title)] = get_pageviews(row.wiki_project, row.wiki_title)
    time.sleep(SLEEP_WIKI)

df_wiki["popularity"] = df_wiki.apply(lambda row: pageview_map.get((row.wiki_project, row.wiki_title), 0), axis=1).astype(int)

# Prefer Wikidata P18 images. Query Wikipedia thumbnails only as a fallback.
thumbnail_map = {}
missing_image_articles = df_wiki.loc[df_wiki["wikidata_image_url"].isna(), ["wiki_project", "wiki_title"]].drop_duplicates()
for row in tqdm(missing_image_articles.itertuples(index=False), total=len(missing_image_articles), desc="Wikipedia thumbnails"):
    thumbnail_map[(row.wiki_project, row.wiki_title)] = get_wikipedia_thumbnail(row.wiki_project, row.wiki_title)
    time.sleep(SLEEP_WIKI)

df_wiki["wikipedia_thumbnail_url"] = df_wiki.apply(lambda row: thumbnail_map.get((row.wiki_project, row.wiki_title)), axis=1)
df_wiki["image_url"] = df_wiki["wikidata_image_url"].fillna(df_wiki["wikipedia_thumbnail_url"])
df_wiki["image_source"] = None
df_wiki.loc[df_wiki["wikidata_image_url"].notna(), "image_source"] = "Wikidata P18"
df_wiki.loc[df_wiki["wikidata_image_url"].isna() & df_wiki["wikipedia_thumbnail_url"].notna(), "image_source"] = "Wikipedia thumbnail"
print(f"Median views: {int(df_wiki.popularity.median()):,}")
print(f"Zero-views: {(df_wiki.popularity == 0).sum()} species")
print(f"Images found: {df_wiki.image_url.notna().sum():,} / {len(df_wiki):,}")
df_wiki.popularity.describe()

---
## 5 · Label selection & GeoJSON export

In [ ]:
# Label: prefer main_common_name, fall back to scientific_name, then IUCN taxon ID.
common_label = df_wiki["main_common_name"].notna() & (df_wiki["main_common_name"] != "")
scientific_label = df_wiki["scientific_name"].notna() & (df_wiki["scientific_name"] != "")
df_wiki["label"] = "IUCN taxon " + df_wiki["taxonid"].astype(str)
df_wiki.loc[scientific_label, "label"] = df_wiki.loc[scientific_label, "scientific_name"]
df_wiki.loc[common_label, "label"] = df_wiki.loc[common_label, "main_common_name"].str.title()

# Optional: drop species with 0 views (no Wikipedia presence → won't render usefully)
# df_wiki = df_wiki[df_wiki.popularity > 0]

print(f"Final taxon count: {len(df_wiki):,}")
df_wiki[["label", "category_iucn", "wiki_title", "wiki_language", "wiki_project", "image_source", "image_url", "population_trend", "number_of_mature_individuals", "estimated_area_of_occupancy", "estimated_extent_of_occurrence", "computed_range_area_km2", "computed_range_component_area_km2", "spatial_presence_label", "spatial_seasonal_label", "taxon_class", "taxon_group", "assessment_date", "year_published", "centroid_source", "centroid_rank", "centroid_count", "popularity", "lon", "lat"]].sample(min(10, len(df_wiki)))

In [ ]:
# Build lightweight centroid GeoJSON used by the browser
def clean_json_value(value):
    """Convert pandas/numpy nulls and scalar values into JSON-safe Python values."""
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(value, "item") and not isinstance(value, str):
        try:
            return value.item()
        except Exception:
            pass
    return value


def feature_properties(row):
    """Build the shared GeoJSON properties for centroid and spatial features."""
    return {
        "taxonid": int(row.taxonid),
        "assessment_id": int(row.assessment_id),
        "assessment_date": clean_json_value(row.assessment_date),
        "year_published": clean_json_value(row.year_published),
        "iucn_assessment_url": clean_json_value(row.iucn_assessment_url),
        "iucn_citation": clean_json_value(row.iucn_citation),
        "wiki_title": clean_json_value(row.wiki_title),
        "wiki_language": clean_json_value(row.wiki_language),
        "wiki_project": clean_json_value(row.wiki_project),
        "wiki_url": clean_json_value(row.wiki_url),
        "wikidata_image_url": clean_json_value(row.wikidata_image_url),
        "wikipedia_thumbnail_url": clean_json_value(row.wikipedia_thumbnail_url),
        "image_url": clean_json_value(row.image_url),
        "image_source": clean_json_value(row.image_source),
        "label": row.label,
        "category_iucn": row.category_iucn,
        "population_trend": clean_json_value(row.population_trend),
        "number_of_mature_individuals": clean_json_value(row.number_of_mature_individuals),
        "estimated_area_of_occupancy": clean_json_value(row.estimated_area_of_occupancy),
        "estimated_extent_of_occurrence": clean_json_value(row.estimated_extent_of_occurrence),
        "taxon_class": row.taxon_class,
        "taxon_group": row.taxon_group,
        "taxon_rank": clean_json_value(row.taxon_rank),
        "parent_taxonid": clean_json_value(row.parent_taxonid),
        "child_infrarank_taxonids": clean_json_value(row.child_infrarank_taxonids),
        "iucn_has_ranges": clean_json_value(row.iucn_has_ranges),
        "iucn_has_points": clean_json_value(row.iucn_has_points),
        "centroid_source": clean_json_value(row.centroid_source),
        "centroid_rank": clean_json_value(row.centroid_rank),
        "centroid_count": clean_json_value(row.centroid_count),
        "range_component_count": clean_json_value(row.range_component_count),
        "spatial_presence": clean_json_value(row.spatial_presence),
        "spatial_presence_label": clean_json_value(row.spatial_presence_label),
        "spatial_seasonal": clean_json_value(row.spatial_seasonal),
        "spatial_seasonal_label": clean_json_value(row.spatial_seasonal_label),
        "computed_range_area_km2": clean_json_value(row.computed_range_area_km2),
        "computed_range_component_area_km2": clean_json_value(row.computed_range_component_area_km2),
        "range_component_area_km2": clean_json_value(row.range_component_area_km2),
        "observation_point_count": clean_json_value(row.observation_point_count),
        "source_paths": clean_json_value(row.source_paths),
        "spatial_citation": clean_json_value(row.spatial_citation),
        "spatial_year": clean_json_value(row.spatial_year),
        "spatial_credit": clean_json_value(row.spatial_credit),
        "iucn_dataset_citation": clean_json_value(row.iucn_dataset_citation),
        "iucn_data_last_updated": clean_json_value(row.iucn_data_last_updated),
        "popularity": int(row.popularity),
    }


features = []
for _, row in df_wiki.iterrows():
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [round(row.lon, 4), round(row.lat, 4)]
        },
        "properties": feature_properties(row)
    })

geojson = {"type": "FeatureCollection", "features": features}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Written: {OUTPUT_PATH} — {len(features):,} centroid features, {size_mb:.1f} MB")

# Build full spatial GeoJSON as a sidecar. This can be much heavier than the
# browser-facing point file, so keep it separate.
spatial_features = []
species_props_df = df_wiki.drop_duplicates(subset="taxonid").copy()
spatial_df = species_props_df.merge(spatial_geometries, on="taxonid", how="inner")
for _, row in spatial_df.iterrows():
    if row.geometry is None or row.geometry.is_empty:
        continue
    props = feature_properties(row)
    props["spatial_source"] = row.spatial_source
    props["spatial_geometry_type"] = row.geometry.geom_type
    spatial_features.append({
        "type": "Feature",
        "geometry": mapping(row.geometry),
        "properties": props
    })

spatial_geojson = {"type": "FeatureCollection", "features": spatial_features}
with open(SPATIAL_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(spatial_geojson, f, ensure_ascii=False)

spatial_size_mb = os.path.getsize(SPATIAL_OUTPUT_PATH) / 1e6
print(f"Written: {SPATIAL_OUTPUT_PATH} — {len(spatial_features):,} spatial features, {spatial_size_mb:.1f} MB")

---
## Quick sanity check

In [ ]:
# Load back and inspect
with open(OUTPUT_PATH) as f:
    check = json.load(f)

sample = check["features"][:5]
for feat in sample:
    p = feat["properties"]
    c = feat["geometry"]["coordinates"]
    year = str(p.get("year_published") or "Unknown")
    print(f"{p['label']:40s} [{p['category_iucn']}] {p.get('taxon_class', 'Unknown'):16s} → {p.get('taxon_group', 'Unknown'):28s} assessed {year:>7s} {p.get('centroid_source', 'unknown')} #{p.get('centroid_rank', '?')}/{p.get('centroid_count', '?')} {p['popularity']:>10,} views  ({c[1]:.2f}, {c[0]:.2f})")